## Bronze Layer

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date

# =========================================================
# Configuration
# =========================================================
raw_path = "/Volumes/fintech/bronze/raw_files/"

checkpoint_path = "/Volumes/fintech/bronze/checkpoints/stock_prices/"
schema_path = "/Volumes/fintech/bronze/schemas/stock_prices/"

# =========================================================
# Auto Loader - Incremental ingestion
# =========================================================
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.includeExistingFiles", "false")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(raw_path)
)

# =========================================================
# Bronze transformations (Apply Schema)
# =========================================================
df_bronze = (
    df_raw
    .select(
        col("symbol")
            .cast("string")
            .alias("symbol"),

        to_date(col("date"))
            .alias("trade_date"),

        col("open")
            .cast("decimal(18,4)")
            .alias("open"),

        col("high")
            .cast("decimal(18,4)")
            .alias("high"),

        col("low")
            .cast("decimal(18,4)")
            .alias("low"),

        col("close")
            .cast("decimal(18,4)")
            .alias("close"),

        col("volume")
            .cast("long")
            .alias("volume"),

        col("change")
            .cast("decimal(18,4)")
            .alias("change"),

        col("changePercent")
            .cast("decimal(18,6)")
            .alias("change_percent"),

        col("vwap")
            .cast("decimal(18,4)")
            .alias("vwap"),

        current_timestamp()
            .alias("_ingestion_timestamp"),

        col("_metadata.file_path")
            .alias("_source_file"),
    )
)

# =========================================================
# Write to Bronze
# =========================================================
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("fintech.bronze.stock_prices")
)

query.awaitTermination()